In [1]:
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
import os
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
class Dataset(Dataset):
    def __init__(self,image_path,mask_path,transform=None):
        self.image_path=image_path
        self.mask_path=mask_path
        self.transform=transform
        self.images=os.listdir(image_path)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
            img_path = os.path.join(self.image_path, self.images[idx])
            mask_path = os.path.join(self.mask_path, self.images[idx])
            
            image = Image.open(img_path).convert('RGB')
            mask = Image.open(mask_path).convert('L')
            fixed_size = (256, 256)
            image = image.resize(fixed_size)
            mask = mask.resize(fixed_size)
            mask = np.array(mask)
            mask = np.where(mask > 0, 1, 0)
            mask = torch.tensor(mask, dtype=torch.long)
            
            if self.transform is not None:
                image = self.transform(image)
            
            return image, mask

        

transform = transforms.Compose([
         transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
dataset = Dataset(r"D:\COMPUTER_VISION\anomaly_detection_test_data\anomaly_detection_test_data\bad", r"D:\COMPUTER_VISION\anomaly_detection_test_data\anomaly_detection_test_data\masks", transform=transform)
dataloader=torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)

In [3]:
import torchvision.models.segmentation as models 
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights
model = models.deeplabv3_resnet50(pretrained=False)
model = models.deeplabv3_resnet50(weights=None)

C:\Users\wwwra\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\wwwra\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [4]:
num_classes = 2 
model.classifier = nn.Sequential(
    nn.Conv2d(2048, 256, kernel_size=3, padding=1, bias=False),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.Conv2d(256, 1, kernel_size=1) 
)

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [6]:
import torch.nn.functional as F
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss
criterion = FocalLoss(alpha=0.25, gamma=2.0, reduction='mean')

In [7]:
optimizer = optim.SGD(model.parameters(), lr=1e-4)


In [8]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()  
        outputs = model(images)['out'] 

        masks = masks.unsqueeze(1).float()  
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / len(dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")


Epoch [1/10], Loss: 0.0354
Epoch [2/10], Loss: 0.0322
Epoch [3/10], Loss: 0.0297
Epoch [4/10], Loss: 0.0275
Epoch [5/10], Loss: 0.0258
Epoch [6/10], Loss: 0.0242
Epoch [7/10], Loss: 0.0229
Epoch [8/10], Loss: 0.0217
Epoch [9/10], Loss: 0.0207
Epoch [10/10], Loss: 0.0198


In [9]:
import torch

def pixel_accuracy(output, mask):
    output = torch.sigmoid(output) 
    pred = (output > 0.5).float()  
    correct = (pred == mask).float()
    acc = correct.sum() / correct.numel()
    return acc.item()

def iou_score(output, mask):
    output = torch.sigmoid(output)
    pred = (output > 0.5).float()
    intersection = (pred * mask).sum()
    union = pred.sum() + mask.sum() - intersection + 1e-6
    return (intersection / union).item()

def dice_score(output, mask):
    output = torch.sigmoid(output)
    pred = (output > 0.5).float()
    intersection = (pred * mask).sum()
    return (2 * intersection / (pred.sum() + mask.sum() + 1e-6)).item()


In [12]:
model.eval()

total_acc = 0.0
total_iou = 0.0
total_dice = 0.0
count = 0

with torch.no_grad():
    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)
        outputs = model(images)['out']        
        masks = masks.unsqueeze(1).float()     
        acc = pixel_accuracy(outputs, masks)
        iou = iou_score(outputs, masks)
        dice = dice_score(outputs, masks)

        total_acc += acc
        total_iou += iou
        total_dice += dice
        count += 1


final_acc = total_acc / count
final_iou = total_iou / count
final_dice = total_dice / count

print("\n======= Final Metrics After Training =======")
print(f"Pixel Accuracy: {final_acc:.4f}")
print(f"IoU Score:      {final_iou:.4f}")
print(f"Dice Score:     {final_dice:.4f}")



===== Final Metrics After Training =====
Pixel Accuracy: 0.9465
IoU Score:      0.0565
Dice Score:     0.1015


In [13]:
import torch

torch.save(model, "deeplabv3_full.pth")


In [14]:
torch.save(model.state_dict(), "deeplabv3_weights.pth")